# Notebook 2 — Region-Level Tissue Reasoning

Demonstrates how to:
1. Extract patch-level appearance features for each detected cell.
2. Assign each cell a tissue-type label from the segmentation mask.
3. Build local tissue-context histograms.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

from src.tissue_reasoning.region_extractor import RegionExtractor
from src.tissue_reasoning.tissue_classifier import TissueClassifier

In [ ]:
# ------------------------------------------------------------------
# Paths — update these to point at your data
# ------------------------------------------------------------------
DETECTION_CSV = "path/to/cell_detections.csv"
TISSUE_MASK   = "path/to/tissue_mask.png"
WSI_IMAGE     = "path/to/wsi_region.png"   # RGB image (or extracted tile)

NUM_TISSUE_CLASSES = 5

In [ ]:
# Load data
detections  = pd.read_csv(DETECTION_CSV)
cell_coords = detections[["x", "y"]].values.astype(np.float32)
wsi_image   = np.array(Image.open(WSI_IMAGE).convert("RGB"))
tissue_mask = np.array(Image.open(TISSUE_MASK))

print(f"Cells: {len(cell_coords)} | Image: {wsi_image.shape} | Mask: {tissue_mask.shape}")

In [ ]:
# Feature extraction
extractor = RegionExtractor(backbone="resnet50", pretrained=True, patch_size=64)
features  = extractor.extract_cell_patches(wsi_image, cell_coords)
print(f"Feature matrix shape: {features.shape}")

In [ ]:
# Tissue-type assignment
classifier     = TissueClassifier(num_classes=NUM_TISSUE_CLASSES)
tissue_labels  = classifier.assign_tissue_labels(tissue_mask, cell_coords)
tissue_context = classifier.build_region_context(tissue_mask, cell_coords, radius=32)

print(f"Tissue label distribution: {dict(zip(*np.unique(tissue_labels, return_counts=True)))}")

In [ ]:
# Visualise tissue context histograms (mean per tissue type)
fig, ax = plt.subplots(figsize=(8, 4))
ax.imshow(tissue_context[:100], aspect="auto", cmap="viridis")
ax.set_xlabel("Tissue class")
ax.set_ylabel("Cell index (first 100)")
ax.set_title("Local tissue-context histograms")
plt.colorbar(ax.images[0], ax=ax, label="Frequency")
plt.tight_layout()
plt.show()